# 03 — Causal ML: efectos heterogéneos (CATE)

## Marco conceptual

En el notebook 02 estimamos el **ATE** (efecto promedio): ¿funciona el nudge *en general*?

Aquí estimamos el **CATE** (efecto condicional): ¿funciona el nudge *para clientes con perfil X*?

\[
\tau(x) = \mathbb{E}[Y(1) - Y(0) \mid X = x]
\]

**Meta-learners** (S/T/X-Learner) y **LinearDML** estiman efectos heterogéneos. En un RCT la propensión es ~0.5, lo que simplifica la identificación.

> **Guía completa:** [`docs/GUIA_CONCEPTUAL_TECNICA.md`](../docs/GUIA_CONCEPTUAL_TECNICA.md) y [`docs/CAUSAL_ML.md`](../docs/CAUSAL_ML.md).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.causal import (
    calibrate_cate_to_ate,
    fit_cate,
    prep_binary_comparison,
    segment_cate_summary,
    validate_cate_vs_ate,
)
from src.data import load_data
from src.mediation import all_funnel_mediations

sns.set_theme(style='whitegrid')
df = load_data(ROOT / 'data' / 'datos_prueba_tecnica.csv')


## Preparación: comparaciones binarias tratamiento vs control

In [ ]:
X1, T1, Y1 = prep_binary_comparison(df, 'trat1')
X2, T2, Y2 = prep_binary_comparison(df, 'trat2')
print(f'Trat1 vs ctrl: {X1.shape[0]} obs, {X1.shape[1]} features')
print(f'Trat2 vs ctrl: {X2.shape[0]} obs, {X2.shape[1]} features')

## S/T/X-Learner y LinearDML (EconML)

In [ ]:
cate_trat1 = fit_cate(X1, T1, Y1, 'trat1 vs ctrl (ctor)')
cate_trat2 = fit_cate(X2, T2, Y2, 'trat2 vs ctrl (ctor)')

cols = ['cate_s', 'cate_t', 'cate_x', 'cate_dml', 'cate_cf']
cate_trat1.to_frame().groupby('label')[cols].mean().round(4)


## Distribución de CATE

In [ ]:
cate_all = pd.concat([cate_trat1.to_frame(), cate_trat2.to_frame()], ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, label in zip(axes, cate_all['label'].unique()):
    subset = cate_all[cate_all['label'] == label]
    sns.kdeplot(subset['cate_x'], fill=True, ax=ax, label='X-Learner')
    ax.axvline(subset['cate_x'].mean(), color='red', linestyle='--', label='Media CATE')
    ax.set_title(label)
    ax.set_xlabel('CATE estimado (ctor)')
    ax.legend()
plt.tight_layout()
plt.show()

## Mediación del funnel (`or` → `ctor`)

Como `ctor` está anidado en `or`, descomponemos el ATE en **vía apertura** vs **vía conversión post-apertura**. Detalle en [`docs/CAUSAL_ML.md`](../docs/CAUSAL_ML.md) §8.


In [ ]:
med = all_funnel_mediations(df)
med[['comparison', 'ate_ctor', 'effect_via_open', 'effect_via_conversion',
     'share_via_open', 'share_via_conversion']].round(4)


## Heterogeneidad por segmentos

In [ ]:
print('Trat1 — por edad:')
display(segment_cate_summary(cate_trat1.to_frame(), 'edad', bins=[18, 35, 50, 100], labels=['18-35', '36-50', '51+']))
print('Trat2 — por uso app:')
display(segment_cate_summary(cate_trat2.to_frame(), 'uso_app'))

**Validación:** La media del CATE debería aproximar el ATE del notebook 02. Si hay brecha (común con RF en outcomes binarios), usar CATE para **ranking de segmentos**, no para magnitudes absolutas. Opcional: `calibrate_cate_to_ate` alinea la media al ATE. Ver [`docs/CAUSAL_ML.md`](../docs/CAUSAL_ML.md).


In [ ]:
validate_cate_vs_ate(df, 'trat2', 'ctor', cate_trat2).round(4)